# Clase 13-Oct — Representaciones Vectoriales de Palabras: del One-Hot al Embedding

Esta sesión conecta todo lo visto en el curso:
- **Lab #1–3:** Bag-of-Words, TF-IDF, One-Hot
- **Lab #5:** N-gramas y matrices de co-ocurrencia
- **Lab #7–8:** `nn.Embedding` en PyTorch
- **Hoy:** El puente entre esas ideas y las librerías modernas de embeddings

Cubriremos **cinco estrategias** en orden histórico:

| # | Estrategia | Librería | Año |
|---|---|---|---|
| 1 | Matriz de Co-ocurrencia (PMI) | `numpy` + `scipy` | ~1990 |
| 2 | Word2Vec | `gensim` | 2013 |
| 3 | GloVe (vectores pre-entrenados) | `gensim` / manual | 2014 |
| 4 | FastText (subpalabras) | `gensim` | 2016 |
| 5 | Embeddings contextuales con spaCy | `spacy` | 2017+ |

> **Nota:** Si `gensim` no está instalado en tu entorno, ejecutá `pip install gensim` en el terminal con el env `nlp-ufm` activo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import Counter, defaultdict
import math
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

print("Importaciones base: OK")

---
## 1. El punto de partida: por qué One-Hot falla

Recordemos el problema central. Con One-Hot, cada palabra es un vector ortogonal a todas las demás.
La similitud entre cualquier par de palabras distintas es **siempre 0**, sin importar su significado.

In [ ]:
# ── Vocabulario de ejemplo ──────────────────────────────────────────────────
vocab = ["rey", "reina", "hombre", "mujer", "perro", "gato", "ladra", "maulla"]
word2id = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

def one_hot(palabra):
    v = np.zeros(V)
    v[word2id[palabra]] = 1.0
    return v

def coseno(a, b):
    norma = np.linalg.norm(a) * np.linalg.norm(b)
    return np.dot(a, b) / norma if norma > 0 else 0.0

# ── Similitudes con One-Hot ─────────────────────────────────────────────────
pares = [("rey", "reina"), ("rey", "hombre"), ("perro", "gato"), ("rey", "perro")]

print("Similitud coseno con ONE-HOT:")
print("-" * 40)
for a, b in pares:
    sim = coseno(one_hot(a), one_hot(b))
    print(f"  {a:8} ↔ {b:8}  →  {sim:.4f}")

print("\n→ Todos los pares tienen similitud 0. El modelo no sabe NADA sobre relaciones semánticas.")

---
## 2. Estrategia 1 — Matriz de Co-ocurrencia con PMI

### Idea
Contamos cuántas veces cada par de palabras aparece dentro de una **ventana de contexto** (±k palabras).
Luego pesamos con **PMI (Pointwise Mutual Information)**:

$$\text{PMI}(w, c) = \log \frac{P(w, c)}{P(w) \cdot P(c)}$$

- PMI alto → la palabra $w$ y el contexto $c$ co-ocurren **más de lo esperado por azar**
- PMI < 0 → se ven juntas **menos de lo esperado** (se reemplaza por 0 en PPMI)

Esto es esencialmente lo mismo que el `indice_bi` del Lab #5, pero simétrico y normalizado.

In [ ]:
# ── Corpus pequeño de ejemplo ───────────────────────────────────────────────
corpus = [
    "el rey tiene poder",
    "la reina tiene poder",
    "el rey gobierna el reino",
    "la reina gobierna el reino",
    "el hombre trabaja duro",
    "la mujer trabaja duro",
    "el hombre come pan",
    "la mujer come pan",
    "el perro ladra fuerte",
    "el gato maulla suave",
    "el perro corre rapido",
    "el gato corre rapido",
    "el rey es un hombre poderoso",
    "la reina es una mujer poderosa",
]

# Construir vocabulario
tokens_todos = [t for frase in corpus for t in frase.split()]
vocab_full = sorted(set(tokens_todos))
w2i = {w: i for i, w in enumerate(vocab_full)}
V_full = len(vocab_full)

print(f"Vocabulario ({V_full} palabras): {vocab_full}")

In [ ]:
# ── Construir matriz de co-ocurrencia ──────────────────────────────────────
VENTANA = 2  # ±2 palabras

cooc = np.zeros((V_full, V_full))

for frase in corpus:
    tokens = frase.split()
    for i, palabra in enumerate(tokens):
        for j in range(max(0, i - VENTANA), min(len(tokens), i + VENTANA + 1)):
            if i != j:
                cooc[w2i[palabra], w2i[tokens[j]]] += 1

# ── Calcular PPMI (Positive PMI) ───────────────────────────────────────────
total = cooc.sum()
p_palabra = cooc.sum(axis=1) / total   # P(w)
p_ctx     = cooc.sum(axis=0) / total   # P(c)
p_joint   = cooc / total               # P(w, c)

ppmi = np.zeros_like(cooc)
for i in range(V_full):
    for j in range(V_full):
        if p_joint[i, j] > 0 and p_palabra[i] > 0 and p_ctx[j] > 0:
            pmi_val = math.log2(p_joint[i, j] / (p_palabra[i] * p_ctx[j]))
            ppmi[i, j] = max(0, pmi_val)  # PPMI: recortar negativos a 0

print("Matriz PPMI construida.")
print(f"Shape: {ppmi.shape} — cada fila es el vector de una palabra ({V_full} dims)\n")

# ── Similitudes con PPMI ───────────────────────────────────────────────────
palabras_foco = ["rey", "reina", "hombre", "mujer", "perro", "gato"]
pares_eval = [("rey", "reina"), ("rey", "hombre"), ("perro", "gato"), ("rey", "perro")]

print("Similitud coseno con PPMI (co-ocurrencia):")
print("-" * 45)
for a, b in pares_eval:
    if a in w2i and b in w2i:
        sim = coseno(ppmi[w2i[a]], ppmi[w2i[b]])
        print(f"  {a:8} ↔ {b:8}  →  {sim:.4f}")

print("\n→ Ahora palabras semánticamente relacionadas tienen similitud > 0.")

In [ ]:
# ── Heatmap de la sub-matriz para las palabras de interés ──────────────────
idx = [w2i[w] for w in palabras_foco if w in w2i]
etiquetas = [vocab_full[i] for i in idx]
submat = ppmi[np.ix_(idx, idx)]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(submat, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(etiquetas)))
ax.set_yticks(range(len(etiquetas)))
ax.set_xticklabels(etiquetas, rotation=45, ha='right')
ax.set_yticklabels(etiquetas)
plt.colorbar(im, ax=ax, label='PPMI')
for i in range(len(etiquetas)):
    for j in range(len(etiquetas)):
        ax.text(j, i, f"{submat[i,j]:.1f}", ha='center', va='center', fontsize=8,
                color='white' if submat[i,j] > submat.max()*0.6 else 'black')
ax.set_title("Sub-matriz PPMI — Co-ocurrencia de palabras clave", fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nObservá: rey↔reina y perro↔gato tienen valores altos (contextos similares).")
print("rey↔perro tiene valor bajo (contextos distintos).")

### El problema que queda: dimensionalidad y dispersión

La matriz PPMI tiene forma $(|V|, |V|)$. Con el vocabulario del Lab #3 (16,131 palabras):  
- **260 millones de celdas** por matriz
- La mayoría son **ceros** (corpus disperso)
- Los vectores no capturan relaciones de segundo orden (sinónimos que nunca co-ocurren directamente)

**Solución histórica:** Truncated SVD / LSA — comprimir la matriz a $d$ dimensiones.  
**Solución moderna:** entrenar una red neuronal que aprenda a comprimir → Word2Vec.

In [ ]:
# ── SVD truncada sobre PPMI (equivalente a LSA/LSI) ─────────────────────────
from numpy.linalg import svd

D = 2   # reducir a 2 dimensiones para poder visualizar

# SVD completa y truncada manualmente
U, s, Vt = svd(ppmi, full_matrices=False)
embeddings_svd = U[:, :D] * s[:D]   # proyección en las D componentes principales

# Graficar solo las palabras de interés
fig, ax = plt.subplots(figsize=(7, 5))
colores = {'rey': '#e74c3c', 'reina': '#e74c3c', 'hombre': '#3498db',
           'mujer': '#3498db', 'perro': '#27ae60', 'gato': '#27ae60'}

for palabra in palabras_foco:
    if palabra in w2i:
        i = w2i[palabra]
        x, y = embeddings_svd[i, 0], embeddings_svd[i, 1]
        color = colores.get(palabra, '#888')
        ax.scatter(x, y, color=color, s=120, zorder=3)
        ax.annotate(palabra, (x, y), textcoords='offset points',
                    xytext=(6, 4), fontsize=11)

ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_title("Embeddings via SVD sobre PPMI (LSA) — 2 dimensiones", fontsize=11, fontweight='bold')
ax.set_xlabel("Componente 1")
ax.set_ylabel("Componente 2")
plt.tight_layout()
plt.show()

print("Palabras del mismo color comparten contextos similares → deberían agruparse.")

---
## 3. Estrategia 2 — Word2Vec con `gensim`

Word2Vec no construye la matriz de co-ocurrencia explícitamente.
En cambio, entrena una red neuronal de **una sola capa oculta** para predecir:
- **Skip-gram:** dada la palabra central → predecir las palabras del contexto
- **CBOW:** dado el contexto → predecir la palabra central

Los **pesos de la capa de embedding** son el resultado que buscamos.

In [ ]:
try:
    from gensim.models import Word2Vec
    GENSIM_OK = True
    print("gensim disponible ✓")
except ImportError:
    GENSIM_OK = False
    print("gensim no está instalado. Instalá con: pip install gensim")
    print("Las celdas de Word2Vec mostrarán el código pero no ejecutarán.")

In [ ]:
if GENSIM_OK:
    # ── Preparar corpus tokenizado ──────────────────────────────────────────
    corpus_w2v = [frase.split() for frase in corpus]

    # ── Entrenar Word2Vec Skip-gram ─────────────────────────────────────────
    modelo_w2v = Word2Vec(
        sentences=corpus_w2v,
        vector_size=10,   # 10 dims (corpus pequeño; en producción: 100-300)
        window=2,         # ventana de contexto ±2 — igual que nuestra matriz manual
        min_count=1,      # incluir palabras con al menos 1 aparición
        sg=1,             # 1 = Skip-gram, 0 = CBOW
        epochs=200,
        seed=42
    )

    print("Modelo Word2Vec entrenado.")
    print(f"Vocabulario: {list(modelo_w2v.wv.key_to_index.keys())}")
    print(f"\nVector de 'rey' (10 dimensiones):")
    print(np.round(modelo_w2v.wv['rey'], 3))

In [ ]:
if GENSIM_OK:
    # ── Similitudes Word2Vec ────────────────────────────────────────────────
    print("Similitud coseno con WORD2VEC (Skip-gram):")
    print("-" * 45)
    for a, b in pares_eval:
        if a in modelo_w2v.wv and b in modelo_w2v.wv:
            sim = modelo_w2v.wv.similarity(a, b)
            print(f"  {a:8} ↔ {b:8}  →  {sim:.4f}")

    print("\n── Palabras más cercanas a 'rey': ──")
    for w, s in modelo_w2v.wv.most_similar('rey', topn=4):
        print(f"   {w:12} {s:.4f}")

    print("\n── Analogía: rey - hombre + mujer ≈ ? ──")
    # Con corpus tan pequeño el resultado varía, pero la mecánica es correcta
    try:
        resultado = modelo_w2v.wv.most_similar(
            positive=['rey', 'mujer'], negative=['hombre'], topn=3)
        for w, s in resultado:
            print(f"   {w:12} {s:.4f}")
        print("   (En corpus grande: 'reina' aparecería aquí)")
    except Exception as e:
        print(f"   Error en analogía: {e}")

In [ ]:
if GENSIM_OK:
    # ── Visualización 2D (proyección manual con SVD) ─────────────────────────
    palabras_plot = [w for w in palabras_foco if w in modelo_w2v.wv]
    vecs = np.array([modelo_w2v.wv[w] for w in palabras_plot])

    # Proyección manual a 2D con SVD (equivale a PCA)
    vecs_centered = vecs - vecs.mean(axis=0)
    U2, s2, _ = svd(vecs_centered, full_matrices=False)
    coords = U2[:, :2] * s2[:2]

    fig, ax = plt.subplots(figsize=(7, 5))
    for i, palabra in enumerate(palabras_plot):
        color = colores.get(palabra, '#888')
        ax.scatter(coords[i, 0], coords[i, 1], color=color, s=120, zorder=3)
        ax.annotate(palabra, (coords[i, 0], coords[i, 1]),
                    textcoords='offset points', xytext=(6, 4), fontsize=11)

    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title("Word2Vec Skip-gram — proyección 2D (SVD sobre embeddings)",
                 fontsize=11, fontweight='bold')
    ax.set_xlabel("Componente 1")
    ax.set_ylabel("Componente 2")
    plt.tight_layout()
    plt.show()

### CBOW vs Skip-gram — ¿cuál usar?

| | CBOW | Skip-gram |
|---|---|---|
| Qué hace | contexto → palabra central | palabra central → contexto |
| Velocidad | **más rápido** | más lento |
| Palabras raras | peor | **mejor** (más ejemplos de entrenamiento por palabra) |
| Corpus grande | similar calidad | **mejor** |
| Corpus pequeño | **mejor** | peor |

Regla práctica: corpus < 1M palabras → CBOW. Corpus grande → Skip-gram.

In [ ]:
if GENSIM_OK:
    # ── Comparar CBOW vs Skip-gram en similitudes ───────────────────────────
    modelo_cbow = Word2Vec(
        sentences=corpus_w2v, vector_size=10,
        window=2, min_count=1, sg=0,  # sg=0 → CBOW
        epochs=200, seed=42
    )

    print(f"{'Par':30} {'Skip-gram':>12} {'CBOW':>12}")
    print("-" * 56)
    for a, b in pares_eval:
        sg_sim  = modelo_w2v.wv.similarity(a, b)   if (a in modelo_w2v.wv  and b in modelo_w2v.wv)  else float('nan')
        cb_sim  = modelo_cbow.wv.similarity(a, b)  if (a in modelo_cbow.wv and b in modelo_cbow.wv) else float('nan')
        print(f"  {a:8} ↔ {b:8}         {sg_sim:>10.4f}   {cb_sim:>10.4f}")

---
## 4. Estrategia 3 — GloVe: vectores pre-entrenados

GloVe (Global Vectors, Stanford 2014) resuelve la co-ocurrencia **globalmente** en lugar de por ventanas locales.
Su función objetivo es:

$$J = \sum_{i,j=1}^{|V|} f(X_{ij}) \left( \mathbf{w}_i^T \tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2$$

donde $X_{ij}$ es el conteo de co-ocurrencia de las palabras $i$ y $j$.

La gran ventaja de GloVe en la práctica: **vectores pre-entrenados sobre 6–840 mil millones de tokens**
(Wikipedia + Gigaword + Common Crawl). Podés descargarlos y usarlos directamente.

In [ ]:
if GENSIM_OK:
    import gensim.downloader as api

    # ── Cargar GloVe pre-entrenado (descarga ~66 MB la primera vez) ────────
    # Opciones disponibles:
    #   'glove-wiki-gigaword-50'   →  50 dims, ~66 MB   ← usaremos este
    #   'glove-wiki-gigaword-100'  → 100 dims, ~128 MB
    #   'glove-wiki-gigaword-300'  → 300 dims, ~376 MB
    #   'word2vec-google-news-300' → 300 dims, ~1.6 GB

    print("Cargando GloVe 50d pre-entrenado (Wikipedia + Gigaword)...")
    print("(Descarga ~66 MB la primera vez; luego queda en caché)")
    glove = api.load('glove-wiki-gigaword-50')
    print(f"\nCargado: {len(glove.key_to_index):,} palabras × 50 dimensiones")

In [ ]:
if GENSIM_OK:
    # ── Las analogías clásicas de GloVe ────────────────────────────────────
    analogias = [
        # (positivos, negativos, pregunta)
        (['king', 'woman'], ['man'],   'king - man + woman ≈ ?'),
        (['paris', 'spain'], ['france'], 'paris - france + spain ≈ ?'),
        (['bigger', 'small'], ['big'],   'bigger - big + small ≈ ?'),
        (['walking', 'swam'], ['walked'], 'walking - walked + swam ≈ ?'),
    ]

    print("Analogías vectoriales con GloVe 50d pre-entrenado:")
    print("-" * 55)
    for pos, neg, pregunta in analogias:
        resultado = glove.most_similar(positive=pos, negative=neg, topn=3)
        top = ', '.join([f"{w} ({s:.3f})" for w, s in resultado])
        print(f"  {pregunta}")
        print(f"    → {top}\n")

In [ ]:
if GENSIM_OK:
    # ── Visualización de vecindario semántico ──────────────────────────────
    palabras_es = ['king', 'queen', 'man', 'woman', 'dog', 'cat',
                   'paris', 'france', 'london', 'england']

    vecs_glove = np.array([glove[w] for w in palabras_es])  # shape: (10, 50)

    # Proyección a 2D con SVD
    vc = vecs_glove - vecs_glove.mean(axis=0)
    Ug, sg, _ = svd(vc, full_matrices=False)
    coords_g = Ug[:, :2] * sg[:2]

    grupos = {
        'royalty': (['king', 'queen'], '#e74c3c'),
        'gender':  (['man', 'woman'],   '#3498db'),
        'animals': (['dog', 'cat'],     '#27ae60'),
        'places':  (['paris', 'france', 'london', 'england'], '#9b59b6'),
    }

    fig, ax = plt.subplots(figsize=(8, 6))
    for grupo, (miembros, color) in grupos.items():
        for palabra in miembros:
            i = palabras_es.index(palabra)
            ax.scatter(coords_g[i, 0], coords_g[i, 1], color=color, s=120, zorder=3,
                       label=grupo if palabra == miembros[0] else None)
            ax.annotate(palabra, (coords_g[i, 0], coords_g[i, 1]),
                        textcoords='offset points', xytext=(5, 4), fontsize=10)

    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.axvline(0, color='gray', lw=0.5, ls='--')
    ax.legend(title='Grupo semántico', fontsize=9)
    ax.set_title("GloVe 50d pre-entrenado — clusters semánticos en 2D",
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("Observá que palabras del mismo grupo semántico se agrupan espacialmente.")
    print("Esto emerge solo de la co-ocurrencia — sin ninguna etiqueta manual.")

---
## 5. Estrategia 4 — FastText: embeddings a nivel de subpalabras

FastText (Facebook, 2016) extiende Word2Vec con una idea clave:
en lugar de aprender un vector por **palabra**, aprende vectores por **n-gramas de caracteres**.

El embedding de una palabra es la **suma** de los embeddings de sus n-gramas:

$$\text{emb}(\text{"correr"}) = \text{emb}(\text{"<co"}) + \text{emb}(\text{"cor"}) + \text{emb}(\text{"orr"}) + \text{emb}(\text{"rre"}) + \text{emb}(\text{"rer"}) + \text{emb}(\text{"er>"})$$

Esto resuelve el **problema OOV** del Lab #5 y Lab #7:
- `"correría"` (no vista) → suma de n-gramas de `"correr"` + sufijo `"ía"` → vector razonable
- Captura morfología: `"comer"`, `"comiendo"`, `"comió"` comparten n-gramas → vectores similares

In [ ]:
# ── Demostración manual: cómo FastText descompone en n-gramas ──────────────
def ngrams_caracteres(palabra, n_min=3, n_max=6):
    """Genera n-gramas de caracteres como hace FastText internamente."""
    # FastText agrega marcadores <word> para distinguir inicio/fin
    palabra_marcada = f"<{palabra}>"
    ngrams = set()
    for n in range(n_min, n_max + 1):
        for i in range(len(palabra_marcada) - n + 1):
            ngrams.add(palabra_marcada[i:i+n])
    return ngrams

palabras_demo = ["correr", "corriendo", "correría", "correr", "comer", "comiendo"]

print("N-gramas de caracteres (3 a 6) que FastText usa internamente:\n")
for p in palabras_demo[:3]:
    ng = ngrams_caracteres(p)
    print(f"  '{p}': {sorted(ng)[:8]}... ({len(ng)} n-gramas en total)")

print("\n── Solapamiento de n-gramas (similitud esperada): ──")
pares_ft = [("correr", "corriendo"), ("correr", "correría"), ("correr", "comer")]
for a, b in pares_ft:
    ng_a = ngrams_caracteres(a)
    ng_b = ngrams_caracteres(b)
    jaccard = len(ng_a & ng_b) / len(ng_a | ng_b)
    print(f"  {a:12} ↔ {b:12}  Jaccard(n-gramas): {jaccard:.3f}")

print("\n→ 'correr' y 'corriendo' comparten muchos n-gramas → vectores similares")
print("→ 'correr' y 'correría' también, aunque 'correría' nunca apareció en el corpus")

In [ ]:
if GENSIM_OK:
    from gensim.models import FastText

    # ── Corpus con variaciones morfológicas ────────────────────────────────
    corpus_ft = [
        "el perro corre en el parque",
        "los perros corren en el parque",
        "el gato corrió muy rápido",
        "los gatos correrán mañana",
        "el perro come carne todos los días",
        "los perros comen carne fresca",
        "el gato comió su comida",
        "el hombre camina por el parque",
        "los hombres caminan todos los días",
        "la mujer caminó al mercado",
        "las mujeres caminarán juntas",
        "el niño juega en el parque",
        "los niños juegan con la pelota",
    ]

    corpus_ft_tok = [f.split() for f in corpus_ft]

    modelo_ft = FastText(
        sentences=corpus_ft_tok,
        vector_size=20,
        window=3,
        min_count=1,
        min_n=3,          # n-gramas de al menos 3 caracteres
        max_n=6,          # n-gramas de hasta 6 caracteres
        epochs=200,
        seed=42
    )

    print("FastText entrenado.")
    print("\n── Similitudes entre formas del mismo verbo: ──")
    pares_morfo = [
        ("correr", "corriendo"),
        ("correr", "corrió"),
        ("correr", "correrán"),
        ("comer",  "comiendo"),
        ("perro",  "perros"),
    ]
    for a, b in pares_morfo:
        # FastText puede generar vectores incluso para palabras OOV
        sim = modelo_ft.wv.similarity(a, b)
        en_vocab_a = "(en vocab)" if a in modelo_ft.wv.key_to_index else "(OOV)"
        en_vocab_b = "(en vocab)" if b in modelo_ft.wv.key_to_index else "(OOV)"
        print(f"  {a:12} {en_vocab_a:12} ↔  {b:12} {en_vocab_b:12}  → {sim:.4f}")

    print("\n── Palabras OOV que Word2Vec no podría manejar: ──")
    oov_words = ["correría", "comeríamos", "perrito", "gatito"]
    for oov in oov_words:
        # FastText genera vector usando los n-gramas conocidos
        vec = modelo_ft.wv[oov]   # ← esto no fallaría ni con palabras nunca vistas
        similares = modelo_ft.wv.most_similar(oov, topn=2)
        print(f"  '{oov}' (nunca visto) → más similar a: {[(w,round(s,3)) for w,s in similares]}")

---
## 6. Estrategia 5 — Embeddings estáticos con spaCy

spaCy carga modelos pre-entrenados que incluyen embeddings densos de 96 o 300 dimensiones.
Los modelos `md` y `lg` incluyen vectores; el modelo `sm` del Lab #6 **no los tiene** (usa hash embeddings).

La diferencia clave con Word2Vec/GloVe: spaCy también hace tokenización, POS, NER y dependencias
en el mismo pipeline — el embedding es un componente de ese pipeline.

In [ ]:
try:
    import spacy
    SPACY_OK = True
    print(f"spaCy {spacy.__version__} disponible ✓")
except ImportError:
    SPACY_OK = False
    print("spaCy no disponible.")

if SPACY_OK:
    try:
        nlp = spacy.load('en_core_web_md')  # modelo md = 300d GloVe integrado
        print("Modelo en_core_web_md cargado (300 dims).")
        SPACY_MODEL = 'md'
    except OSError:
        try:
            nlp = spacy.load('en_core_web_sm')  # sm no tiene vectores reales
            print("Cargado en_core_web_sm (sin vectores densos — instala _md o _lg para similitud).")
            SPACY_MODEL = 'sm'
        except OSError:
            SPACY_OK = False
            print("Ningún modelo de spaCy disponible. Instalar con:")
            print("  python -m spacy download en_core_web_md")

In [ ]:
if SPACY_OK and SPACY_MODEL == 'md':
    # ── Similitud semántica vía vectores de spaCy ──────────────────────────
    oraciones = [
        "The cat sat on the mat",
        "A dog rested on the rug",
        "The stock market crashed today",
        "Financial markets declined sharply",
    ]

    docs = [nlp(o) for o in oraciones]

    print("Similitud semántica entre oraciones (spaCy en_core_web_md):")
    print("-" * 65)
    pares_oraciones = [(0,1), (0,2), (0,3), (2,3)]
    for i, j in pares_oraciones:
        sim = docs[i].similarity(docs[j])
        print(f"  [{i}] vs [{j}]:  {sim:.4f}")
        print(f"       '{oraciones[i][:45]}'")
        print(f"       '{oraciones[j][:45]}'\n")

    # ── Embedding de tokens individuales ──────────────────────────────────
    doc_demo = nlp("king queen man woman")
    print("\nVectores de tokens (primeras 8 dimensiones de 300):")
    for token in doc_demo:
        if token.has_vector:
            print(f"  {token.text:8}: {np.round(token.vector[:8], 2)}...")

In [ ]:
if SPACY_OK and SPACY_MODEL == 'md':
    # ── Conexión con el Lab #6: embeddings en POS tagging ─────────────────
    # La palabra 'run' es ambigua: VERB o NOUN según contexto
    # Los embeddings contextuales (próxima clase) lo resuelven mejor,
    # pero spaCy md ya hace algo útil con el contexto del pipeline

    oraciones_amb = [
        "They run fast every morning",
        "She went for a long run in the park",
    ]

    print("POS tagging con spaCy md — desambiguación de 'run':")
    for oracion in oraciones_amb:
        doc_amb = nlp(oracion)
        tokens_info = [(t.text, t.pos_) for t in doc_amb]
        print(f"\n  '{oracion}'")
        for tok, pos in tokens_info:
            marca = " ← 'run'" if tok.lower() == 'run' else ""
            print(f"    {tok:12} {pos}{marca}")

    print("\n→ Mismo resultado que la LSTM del Lab #7, pero usando pipeline preentrenado.")

---
## 7. Implementar Word2Vec desde cero con PyTorch

Esto conecta directamente con el `nn.Embedding` que usaste en el Lab #7 y Lab #8.
La arquitectura Skip-gram es: **One-Hot → Embedding → Softmax → Contexto**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# ── Corpus y vocabulario ───────────────────────────────────────────────────
corpus_pt = [
    "el rey tiene poder y gobierna",
    "la reina tiene poder y gobierna",
    "el rey es un hombre",
    "la reina es una mujer",
    "el hombre trabaja cada dia",
    "la mujer trabaja cada dia",
    "el perro ladra muy fuerte",
    "el gato maulla muy suave",
    "el perro y el gato son animales",
    "el rey y la reina son royalty",
]

tokens_pt = [t for frase in corpus_pt for t in frase.split()]
vocab_pt  = sorted(set(tokens_pt))
w2i_pt    = {w: i for i, w in enumerate(vocab_pt)}
i2w_pt    = {i: w for w, i in w2i_pt.items()}
V_pt      = len(vocab_pt)

print(f"Vocabulario: {V_pt} palabras")
print(vocab_pt)

In [ ]:
# ── Generar pares (palabra_central, contexto) para Skip-gram ───────────────
VENTANA_PT = 2

pares_skipgram = []
for frase in corpus_pt:
    tokens = frase.split()
    for i, centro in enumerate(tokens):
        for j in range(max(0, i - VENTANA_PT), min(len(tokens), i + VENTANA_PT + 1)):
            if i != j:
                pares_skipgram.append((w2i_pt[centro], w2i_pt[tokens[j]]))

print(f"Total de pares entrenamiento: {len(pares_skipgram)}")
print(f"Primeros 5: {pares_skipgram[:5]}")
print(f"  → ({i2w_pt[pares_skipgram[0][0]]}, {i2w_pt[pares_skipgram[0][1]]})")

In [ ]:
# ── Arquitectura Word2Vec Skip-gram en PyTorch ─────────────────────────────
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # Esta es la MISMA capa que usaste en el Lab #7 y Lab #8
        # La diferencia: aquí la entrenamos para predecir contextos,
        # no para clasificar POS tags
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.linear     = nn.Linear(embed_dim, vocab_size)

    def forward(self, centro):
        # centro: índice de la palabra central (batch de enteros)
        emb = self.embeddings(centro)         # (batch, embed_dim)
        logits = self.linear(emb)             # (batch, vocab_size)
        return logits

    def get_embedding(self, palabra_idx):
        """Extrae el vector aprendido de una palabra."""
        with torch.no_grad():
            return self.embeddings(torch.tensor(palabra_idx)).numpy()


EMBED_DIM = 10
torch.manual_seed(42)
model_sg = SkipGram(V_pt, EMBED_DIM)
optimizer_sg = optim.Adam(model_sg.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# Convertir pares a tensors
centros   = torch.tensor([p[0] for p in pares_skipgram], dtype=torch.long)
contextos = torch.tensor([p[1] for p in pares_skipgram], dtype=torch.long)

# ── Entrenamiento ──────────────────────────────────────────────────────────
EPOCHS = 500
losses = []

for epoch in range(EPOCHS):
    model_sg.train()
    optimizer_sg.zero_grad()
    logits = model_sg(centros)
    loss = loss_fn(logits, contextos)
    loss.backward()
    optimizer_sg.step()
    losses.append(loss.item())

print(f"Entrenamiento finalizado. Loss inicial: {losses[0]:.4f} → Loss final: {losses[-1]:.4f}")

In [ ]:
# ── Curva de pérdida ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color='#3498db', linewidth=1.5)
ax.set_xlabel('Época')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Entrenamiento Word2Vec Skip-gram desde cero (PyTorch)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Extraer embeddings y medir similitudes ─────────────────────────────────
model_sg.eval()

palabras_foco_pt = ["rey", "reina", "hombre", "mujer", "perro", "gato"]
vecs_pt = {w: model_sg.get_embedding(w2i_pt[w]) for w in palabras_foco_pt}

print("Similitudes con embeddings aprendidos desde CERO en PyTorch:")
print("-" * 55)
pares_eval_pt = [("rey", "reina"), ("rey", "hombre"), ("perro", "gato"), ("rey", "perro")]
for a, b in pares_eval_pt:
    sim = coseno(vecs_pt[a], vecs_pt[b])
    print(f"  {a:8} ↔ {b:8}  →  {sim:.4f}")

# ── Visualización 2D ──────────────────────────────────────────────────────
M = np.array([vecs_pt[w] for w in palabras_foco_pt])
Mc = M - M.mean(axis=0)
Up, sp, _ = svd(Mc, full_matrices=False)
coords_pt = Up[:, :2] * sp[:2]

fig, ax = plt.subplots(figsize=(7, 5))
for i, palabra in enumerate(palabras_foco_pt):
    color = colores.get(palabra, '#888')
    ax.scatter(coords_pt[i, 0], coords_pt[i, 1], color=color, s=130, zorder=3)
    ax.annotate(palabra, (coords_pt[i, 0], coords_pt[i, 1]),
                textcoords='offset points', xytext=(6, 4), fontsize=11)

ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.set_title("Skip-gram implementado desde cero con nn.Embedding — 2D",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Comparativa final: los cinco métodos lado a lado

In [ ]:
# ── Tabla comparativa de similitudes por método ────────────────────────────
# (solo los métodos disponibles en el entorno)

resultados = {}

# 1. One-Hot
resultados['One-Hot'] = {(a,b): coseno(one_hot(a), one_hot(b))
                         for a,b in pares if a in w2i and b in w2i}

# 2. PPMI (co-ocurrencia)
resultados['PPMI'] = {(a,b): coseno(ppmi[w2i[a]], ppmi[w2i[b]])
                      for a,b in pares_eval if a in w2i and b in w2i}

# 3. SVD sobre PPMI (LSA)
resultados['LSA/SVD'] = {(a,b): coseno(embeddings_svd[w2i[a]], embeddings_svd[w2i[b]])
                         for a,b in pares_eval if a in w2i and b in w2i}

# 4. PyTorch Skip-gram
resultados['Skip-gram (PyTorch)'] = {(a,b): coseno(vecs_pt[a], vecs_pt[b])
                                      for a,b in pares_eval_pt if a in vecs_pt and b in vecs_pt}

if GENSIM_OK:
    # 5. Word2Vec gensim
    resultados['Word2Vec (gensim)'] = {(a,b): modelo_w2v.wv.similarity(a, b)
                                        for a,b in pares_eval
                                        if a in modelo_w2v.wv and b in modelo_w2v.wv}
    # 6. FastText
    resultados['FastText (gensim)'] = {(a,b): modelo_ft.wv.similarity(a, b)
                                        for a,b in pares_eval
                                        if a in modelo_ft.wv and b in modelo_ft.wv}

print("\n" + "=" * 75)
print("COMPARATIVA DE SIMILITUDES COSENO POR MÉTODO")
print("=" * 75)

par_comun = ("rey", "reina")
par_distinto = ("rey", "perro")

print(f"\n{'Método':<25} {'rey ↔ reina':>15} {'rey ↔ perro':>15}")
print("-" * 58)
for metodo, sims in resultados.items():
    s1 = sims.get(par_comun, sims.get(('rey','reina'), float('nan')))
    s2 = sims.get(par_distinto, sims.get(('rey','perro'), float('nan')))
    print(f"  {metodo:<23} {s1:>13.4f}   {s2:>13.4f}")

print("\n→ Un buen modelo debería tener: rey↔reina ALTO, rey↔perro BAJO")

---
## 9. Resumen: la jerarquía de representaciones

```
┌─────────────────────────────────────────────────────────────────────┐
│                   EVOLUCIÓN DE REPRESENTACIONES                     │
├──────────────┬──────────┬───────────┬───────────┬──────────────────┤
│  Método      │  Dims    │ Semántica │ OOV?      │ Contexto         │
├──────────────┼──────────┼───────────┼───────────┼──────────────────┤
│ One-Hot      │  |V|     │    ❌     │    ❌     │ Ninguno          │
│ BoW/TF-IDF   │  |V|     │    ±      │    ❌     │ Documento        │
│ PPMI         │  |V|     │    ✓      │    ❌     │ Ventana local    │
│ LSA/SVD      │  d<<|V|  │    ✓✓     │    ❌     │ Global           │
│ Word2Vec     │  d<<|V|  │    ✓✓     │    ❌     │ Ventana local    │
│ GloVe        │  d<<|V|  │    ✓✓     │    ❌     │ Global           │
│ FastText     │  d<<|V|  │    ✓✓     │    ✓✓     │ Subpalabras      │
│ BERT/GPT*    │  d<<|V|  │    ✓✓✓    │    ✓✓✓    │ Toda la oración  │
├──────────────┴──────────┴───────────┴───────────┴──────────────────┤
│  * Embeddings CONTEXTUALES — próxima clase (Transformers)           │
└─────────────────────────────────────────────────────────────────────┘
```

### La conexión con tus laboratorios anteriores

| Lab | Qué hiciste | Cómo se conecta con hoy |
|-----|-------------|------------------------|
| #1 / #3 | BoW, TF-IDF, clasificación de noticias | One-Hot de 16,131 dims; la tabla de arriba lo ubica al inicio |
| #5 | N-gramas, `indice_bi`, perplexity | `indice_bi[ctx][palabra]` **es** una fila de la matriz PPMI con ventana=1 |
| #6 | POS tagging con spaCy sm | `sm` usa hash embeddings; `md` usa GloVe integrado |
| #7 | LSTM + `nn.Embedding` | La capa `nn.Embedding` **es exactamente** la matriz W de Word2Vec |
| #8 | PyTorch end-to-end | Aquí entrenamos ese mismo `nn.Embedding` para el objetivo de Skip-gram |